# Bug classifier: the production "suggested bugs" champion (no LLM)
Stage 1 multinomial LR over signature phrases + metadata + text → top-5 candidates → stage 2 pair re-ranker (pair features, base posterior, temporal burst, kNN votes, vendor hard rule) → conformal open-set router (LINK / SIBLINGS / NEW) → cold-start text rule for bugs with no training runs → explanation. Lifecycle: champion/challenger, forward ledger, guarded recalibration.

> All data in this notebook is **synthetic** (generated below). The real system runs on private validation data; the code paths are the same, the numbers in the README are from the real data.

In [ ]:
!pip -q install scikit-learn matplotlib numpy scipy

In [ ]:
%%writefile synthetic.py
"""Synthetic failure generator for the demo notebooks (no real data is published).
Each bug has a signature (2-3 log lines), a domain, an owner and a test family; runs of that bug emit the signature
mixed with background noise, sometimes miss a line, and 'sibling' bugs share most of their signature, which is the
hard case in the real system. Not-a-bug runs are infrastructure or test-content problems."""
import random
from datetime import datetime, timedelta

DOMAINS = ["power", "memory", "reset", "virtualization", "media", "compute", "fabric", "firmware"]
COMPONENTS = ["gt", "dma", "pcie", "fw", "mem", "sched", "link", "irq", "pmu", "vf"]
SYMPTOMS = ["engine reset", "timeout waiting for fence", "page fault at <HEX>", "device lost", "link training failed",
            "ECC uncorrectable", "watchdog expired", "CRC mismatch", "thermal trip", "context banned"]
NOISE = ["test step <N> started", "allocating <N> buffers", "INFO: iteration <N> passed", "loading module", "heartbeat ok",
         "perf counter sample <N>", "config applied", "mounting share", "INFO: retrying connection"]
GENERIC = ["FAIL: test step <N> returned non-zero", "ERROR: timeout waiting for test harness", "FAIL: assertion in test script"]
INFRA = ["ERROR: node unreachable", "ERROR: container image pull failed", "FAIL: missing test binary", "ERROR: disk quota exceeded"]


def make_world(n_bugs=40, n_runs=1200, p_none=.18, seed=0):
    rng = random.Random(seed)
    t0 = datetime(2026, 1, 1)
    bugs = []
    for i in range(n_bugs):
        sib = bugs[rng.randrange(len(bugs))] if bugs and rng.random() < .35 else None
        comp = sib["component"] if sib else rng.choice(COMPONENTS)
        sig = list(sib["signature"][:2]) if sib else [f"{comp}: {rng.choice(SYMPTOMS)}", f"{comp}: {rng.choice(SYMPTOMS)} on <PCI>"]
        sig.append(f"{comp}: first fault code {rng.randrange(16, 255):#x}")
        bugs.append({"id": f"BUG-{i:03d}", "domain": sib["domain"] if sib else rng.choice(DOMAINS), "component": comp,
                     "owner": f"owner{rng.randrange(8)}", "family": f"T{rng.randrange(12)}", "signature": sig,
                     "created": t0 + timedelta(days=rng.randrange(0, 200)),
                     "title": f"[{comp}] {sig[0].split(': ', 1)[1]}", "root_cause": f"{comp} {sig[-1].split(': ', 1)[1]}"})
    w = [1 / (i + 1) ** .8 for i in range(n_bugs)]          # long tail: a few bugs own most failures
    runs = []
    for j in range(n_runs):
        start = t0 + timedelta(days=rng.uniform(0, 260))
        if rng.random() < p_none:
            lines, label, b = ([rng.choice(INFRA)] if rng.random() < .6 else [rng.choice(GENERIC)]), "NONE", None
        else:
            b = rng.choices(bugs, w)[0]
            if b["created"] > start + timedelta(days=30):
                start = b["created"] + timedelta(days=rng.uniform(0, 30))
            lines, label = [s for s in b["signature"] if rng.random() > .3] or b["signature"][:1], b["id"]
            if rng.random() < .25:
                lines.append(rng.choice(GENERIC + INFRA))      # product bugs also trip generic/infra-looking lines
        lines += rng.sample(NOISE, 4)
        rng.shuffle(lines)
        runs.append({"run": f"RUN-{j:05d}", "label": label, "start": start, "lines": lines,
                     "domain": b["domain"] if b else rng.choice(DOMAINS), "owner": b["owner"] if b else f"owner{rng.randrange(8)}",
                     "family": b["family"] if b and rng.random() < .7 else f"T{rng.randrange(12)}"})
    return bugs, runs


def split(runs, bugs, seed=0):
    """Grouped by bug: 30% of bugs are test-only, so the test split measures bugs never seen in training."""
    rng = random.Random(seed)
    ids = [b["id"] for b in bugs]
    rng.shuffle(ids)
    test_bugs = set(ids[: int(.3 * len(ids))])
    for r in runs:
        r["split"] = "test" if (r["label"] in test_bugs or (r["label"] == "NONE" and rng.random() < .3)) else "train"
        # frequent bugs also appear in test (seen-bug slice)
        if r["label"] not in test_bugs and r["label"] != "NONE" and rng.random() < .2:
            r["split"] = "test"
    return runs

In [ ]:
%%writefile ranker.py
"""Failure ranker (no LLM): failed run -> ranked list of known bugs + calibrated LINK / REVIEW / NEW decision.

Features per (run, candidate bug) pair:
  - log channel   : signature lines of the bug found in the run's normalised evidence (exact skeleton hits,
                    fuzzy phrase hits, idf-weighted), first-fault agreement
  - metadata prior: same test family, same domain, same owner, peer votes (how many earlier runs of this test
                    family were linked to the bug), recency of the bug's last link, bug created before the run
  - hard rules    : a bug tagged for another platform vendor never outranks any other candidate
Model: logistic regression over pair features (pairwise -> pointwise scores), grouped CV by bug, then a
calibrated open-set threshold: LINK only when P(top-1) >= tau, tau picked for >= 85 % out-of-fold precision.
"""
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict

import numpy as np
from sklearn.linear_model import LogisticRegression

FEATS = ["sig_hits", "sig_frac", "sig_idf", "first_fault", "same_family", "same_domain", "same_owner",
         "peer_votes", "log_recency", "created_before"]


def tokens(s: str) -> set[str]:
    return set(re.findall(r"[a-z_]{3,}", s.lower()))


class Ranker:
    def __init__(self, C: float = 1.0, pool: int = 50):
        self.C, self.pool = C, pool

    # ------------------------------------------------------------- fit
    def fit(self, runs: list[dict], bugs: list[dict]):
        self.bugs = {b["id"]: b for b in bugs}
        self.df = Counter(t for b in bugs for s in b["signature"] for t in tokens(s))
        self.history = sorted((r for r in runs if r["label"] != "NONE"), key=lambda r: r["start"])
        X, y = [], []
        for r in runs:
            if r["label"] == "NONE":
                continue
            for bid in self._pool(r):
                X.append(self._feat(r, self.bugs[bid]))
                y.append(int(bid == r["label"]))
        self.clf = LogisticRegression(C=self.C, max_iter=2000, class_weight="balanced").fit(np.array(X), np.array(y))
        return self

    # ------------------------------------------------------------- features
    def _pool(self, r):
        return [b for b, v in self.bugs.items() if v["created"] <= r["start"]][: self.pool * 10]

    def _feat(self, r, b):
        text = " ".join(r["lines"]).lower()
        hits = [s for s in b["signature"] if s.lower() in text]
        idf = sum(math.log(1 + len(self.bugs) / (1 + self.df[t])) for s in hits for t in tokens(s))
        prev = [h for h in self.history if h["start"] < r["start"] and h["label"] == b["id"]]
        peers = sum(1 for h in prev if h["family"] == r["family"])
        last = (r["start"] - prev[-1]["start"]).days if prev else 999
        ff = next((l for l in r["lines"] if re.search(r"fault|reset|lost|timeout|mismatch", l)), "")
        return [len(hits), len(hits) / len(b["signature"]), idf, int(bool(ff) and ff in b["signature"]),
                int(b["family"] == r["family"]), int(b["domain"] == r["domain"]), int(b["owner"] == r["owner"]),
                math.log1p(peers), -math.log1p(last), int(b["created"] <= r["start"])]

    # ------------------------------------------------------------- predict
    def rank(self, r, k: int = 5):
        pool = self._pool(r)
        if not pool:
            return []
        p = self.clf.predict_proba(np.array([self._feat(r, self.bugs[b]) for b in pool]))[:, 1]
        order = np.argsort(-p)[:k]
        return [(pool[i], float(p[i])) for i in order]


def calibrate_tau(scored: list[tuple[float, bool]], target_precision: float = .85) -> float:
    """Lowest threshold whose out-of-fold precision on LINK decisions is >= target."""
    for tau in np.arange(.05, 1.0, .01):
        sel = [ok for s, ok in scored if s >= tau]
        if sel and sum(sel) / len(sel) >= target_precision:
            return float(tau)
    return 1.0


def evaluate(ranker: Ranker, runs: list[dict], unseen: set[str] | None = None) -> dict:
    res = defaultdict(list)
    for r in runs:
        if r["label"] == "NONE":
            continue
        top = [b for b, _ in ranker.rank(r, 3)]
        res["top1"].append(bool(top) and top[0] == r["label"])
        res["top3"].append(r["label"] in top)
        if unseen is not None and r["label"] in unseen:
            res["unseen_top1"].append(bool(top) and top[0] == r["label"])
    return {k: round(100 * sum(v) / len(v), 1) for k, v in res.items() if v}

In [ ]:
%%writefile classifier.py
"""Bug classifier: the production champion behind "Suggested bugs" (no LLM).

    failed run ──► stage 1  base classifier      LR over [signature-phrase hits | metadata one-hot | text tf-idf]
               ──► top-K candidates (unthresholded, K=5)
               ──► stage 2  re-ranker            pair LR (ranker.py) + base posterior + temporal burst + kNN votes
               ──► hard rules                    a bug tagged for another platform vendor sinks to the bottom
               ──► open-set router               split-conformal on the absolute top logit: LINK / SIBLINGS / NEW
               ──► cold-start text rule          bug with no training runs whose report text matches strongly
               ──► explanation                   the features that pushed the score (coefficient x value)
    lifecycle: champion/challenger retraining, live refit on new human links, forward ledger + guarded recalibration
"""
from __future__ import annotations

import math
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import timedelta

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import csr_matrix, hstack

from ranker import Ranker

META = ["domain", "owner", "family"]


# ================================================================ stage 1: base classifier
class BaseClassifier:
    """One multinomial LR over three channels. Status, CI result and comments are never features (label leak).
    Failing runs are ranked among bugs only (the NONE column is dropped at prediction time)."""

    def __init__(self, C: float = 100.0, phrase_weight: float = 2.0):
        self.C, self.w = C, phrase_weight

    def _phrases(self, runs):
        text = [" ".join(r["lines"]).lower() for r in runs]
        return csr_matrix([[sum(s.lower() in t for s in b["signature"]) / len(b["signature"]) for b in self.bugs]
                           for t in text]) * self.w

    def _X(self, runs, fit=False):
        meta = [[r.get(k, "") for k in META] for r in runs]
        text = [" ".join(r["lines"]) for r in runs]
        if fit:
            self.oh = OneHotEncoder(handle_unknown="ignore").fit(meta)
            self.tf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(text)
        return hstack([self._phrases(runs), self.oh.transform(meta), self.tf.transform(text)]).tocsr()

    def fit(self, runs, bugs):
        self.bugs = bugs
        self.clf = LogisticRegression(C=self.C, max_iter=3000).fit(self._X(runs, fit=True), [r["label"] for r in runs])
        return self

    def rank(self, run, k=5):
        p = self.clf.predict_proba(self._X([run]))[0]
        out = [(c, float(v)) for c, v in zip(self.clf.classes_, p) if c != "NONE"]
        return sorted(out, key=lambda x: -x[1])[:k]          # unthresholded: low-prior truths stay in the pool

    def why(self, run, bug, top=5):
        """Features pushing `bug` up for this run: coefficient x value (what the UI shows next to a suggestion)."""
        if bug not in self.clf.classes_:
            return []
        x = self._X([run]).toarray()[0]
        contrib = self.clf.coef_[list(self.clf.classes_).index(bug)] * x
        names = [f"phrase:{b['id']}" for b in self.bugs] + list(self.oh.get_feature_names_out(META)) + \
            [f"text:{t}" for t in self.tf.get_feature_names_out()]
        return [(names[i], round(float(contrib[i]), 3)) for i in np.argsort(-contrib)[:top] if contrib[i] > 0]


# ================================================================ stage 2: re-ranker over the candidates
def _jacc(a, b):
    a, b = set(a), set(b)
    return len(a & b) / (len(a | b) or 1)


class Champion:
    """Stage 1 + stage 2. Stage 2 only re-orders stage 1's top-K, so recall@K is its hard ceiling;
    K=5 (instead of 3, and without the 0.02 probability floor) lifted that ceiling by 3-8 points."""

    def __init__(self, K=5, C2=0.1, burst_days=7, knn=7):
        self.K, self.C2, self.burst, self.knn = K, C2, timedelta(days=burst_days), knn

    def _extra(self, r, bid, p_base):
        recent = sum(1 for h in self.hist if h["label"] == bid and r["start"] - self.burst <= h["start"] < r["start"])
        near = sorted((h for h in self.hist if h["start"] < r["start"]), key=lambda h: -_jacc(h["lines"], r["lines"]))[: self.knn]
        votes = sum(h["label"] == bid for h in near) / self.knn
        vendor_clash = int(bool(r.get("vendor")) and bool(self.pair.bugs[bid].get("vendor")) and
                           r["vendor"] != self.pair.bugs[bid]["vendor"])
        return [math.log(p_base + 1e-4), math.log1p(recent), votes, vendor_clash]

    def _rows(self, r, cands):
        return np.array([self.pair._feat(r, self.pair.bugs[b]) + self._extra(r, b, p) for b, p in cands])

    def fit(self, runs, bugs, oof_base: dict | None = None):
        self.base = BaseClassifier().fit(runs, bugs)
        self.pair = Ranker().fit(runs, bugs)
        self.hist = sorted((r for r in runs if r["label"] != "NONE"), key=lambda r: r["start"])
        X, y = [], []
        for r in self.hist:
            cands = (oof_base or {}).get(r["run"]) or self.base.rank(r, self.K)   # out-of-fold when available
            if r["label"] not in [b for b, _ in cands]:
                continue
            X.append(self._rows(r, cands)); y += [int(b == r["label"]) for b, _ in cands]
        self.rr = LogisticRegression(C=self.C2, max_iter=3000, class_weight="balanced").fit(np.vstack(X), y)
        return self

    def rank(self, r):
        cands = self.base.rank(r, self.K)
        logit = self.rr.decision_function(self._rows(r, cands))
        logit = np.where(self._rows(r, cands)[:, -1] > 0, -1e3, logit)   # hard rule: vendor clash goes last
        order = np.argsort(-logit)
        return [(cands[i][0], float(logit[i])) for i in order]


# ================================================================ open-set router (split conformal)
@dataclass
class OpenSetRouter:
    """Softmax-based conformal (1 - p) cannot say NEW: a closed set always has a winner (NEW recall 0%).
    The absolute top logit can (seen-vs-new AUROC .92). Nonconformity = -logit of the true candidate,
    calibrated only on runs whose truth was a candidate. Prediction set = candidates with logit >= -q.
        |set| == 1 -> LINK      |set| > 1 -> SIBLINGS (diff tools / agent)      |set| == 0 -> NEW"""
    alpha: float = .1
    q: float = 0.0

    def calibrate(self, ranked: list[list[tuple[str, float]]], truth: list[str]):
        s = sorted(-dict(rk)[t] for rk, t in zip(ranked, truth) if t in dict(rk))
        n = len(s)
        self.q = s[min(n - 1, math.ceil((n + 1) * (1 - self.alpha)) - 1)]
        return self

    def route(self, ranked):
        keep = [b for b, l in ranked if l >= -self.q]
        return ("LINK" if len(keep) == 1 else "SIBLINGS" if keep else "NEW"), keep


# ================================================================ cold-start text rule
class TextColdStart:
    """Label-free retriever over the bug reports (tf-idf of title + signature + root cause). It fires only for a
    bug that has NO training runs (the classifier cannot know it) and only above theta, the smallest score with
    <= 2% fires on the calibration window, where every such fire is wrong by construction."""

    def fit(self, bugs, trained_bugs: set[str]):
        self.ids = [b["id"] for b in bugs]
        self.cold = {b for b in self.ids if b not in trained_bugs}
        docs = [" ".join([b.get("title", ""), *b["signature"], b.get("root_cause", "")]) for b in bugs]
        self.tf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(docs)
        self.M = self.tf.transform(docs)
        return self

    def top(self, r):
        s = cosine_similarity(self.tf.transform([" ".join(r["lines"])]), self.M)[0]
        i = int(np.argmax(s))
        return self.ids[i], float(s[i])

    def calibrate(self, calib_runs, max_fire=.02, floor=.3):
        """calib_runs: runs whose bug HAS training runs, so any cold fire on them is a false fire."""
        sc = sorted((s for b, s in map(self.top, calib_runs) if b in self.cold), reverse=True)
        k = int(max_fire * len(calib_runs))
        self.theta = max(floor, sc[k] + 1e-6 if len(sc) > k else 0.0)
        return self

    def fire(self, r):
        b, s = self.top(r)
        return b if b in self.cold and s >= self.theta else None


def decide(champ: Champion, router: OpenSetRouter, cold: TextColdStart, r) -> dict:
    """Final answer shown to people: always the top 3 (never hidden), plus the route."""
    ranked = champ.rank(r)
    b = cold.fire(r)
    if b:
        return {"route": "LINK", "bug": b, "top3": [b] + [x for x, _ in ranked[:2]], "why": "cold-start text rule"}
    route, keep = router.route(ranked)
    return {"route": route, "bug": keep[0] if route == "LINK" else None, "top3": [x for x, _ in ranked[:3]],
            "set": keep, "why": champ.base.why(r, ranked[0][0])}


def override_rule(ranked, agent: dict | None, probe: dict[str, float] | None = None, conf_any=.8, conf_second=.7):
    """Live override calibration ("R1"), applied on click when the RCA agent has run: the classifier stays the
    default; the agent's pick replaces it only at confidence >= 0.8, or when the agent picks the classifier's #2
    at >= 0.7 and that bug's probe evidence (share of its signature lines found in this run) is > 0 and at least the
    top's. Measured: 14 fixed / 1 broken, top-1 78.8% -> 81.5%."""
    top = ranked[0][0]
    if not agent or not agent.get("bug"):
        return top
    b, c, probe = agent["bug"], agent.get("confidence", 0), probe or {}
    if c >= conf_any:
        return b
    if len(ranked) > 1 and b == ranked[1][0] and c >= conf_second and probe.get(b, 0) > 0 and probe.get(b, 0) >= probe.get(top, 0):
        return b
    return top


# ================================================================ lifecycle
@dataclass
class Registry:
    """Champion/challenger: a challenger (hyper-parameter grid) is promoted only if it beats the champion on the
    same folds of the newest window. Triggers: bootstrap, new bugs, window rollover, incremental links."""
    versions: list = field(default_factory=list)

    def consider(self, name, score, params):
        best = max((v["score"] for v in self.versions if v["promoted"]), default=-1)
        v = {"name": name, "score": score, "params": params, "promoted": score > best}
        self.versions.append(v)
        return v["promoted"]


@dataclass
class Ledger:
    """Forward check (the drift alarm). Every day: record the live answer for failed runs nobody has linked yet;
    later, score those predictions against the links people actually made."""
    rows: dict = field(default_factory=dict)

    def record(self, run_id, route, bug):
        self.rows.setdefault(run_id, {"route": route, "bug": bug, "truth": None})

    def resolve(self, links: dict[str, str]):
        for rid, b in links.items():
            if rid in self.rows:
                self.rows[rid]["truth"] = b

    def forward(self):
        done = [v for v in self.rows.values() if v["truth"] and v["route"] == "LINK"]
        prec = sum(v["bug"] == v["truth"] for v in done) / len(done) if done else None
        return {"linked": len(done), "link_precision": prec}

    def needs_recalibration(self, floor=.85, min_n=10):
        f = self.forward()
        return f["linked"] >= min_n and f["link_precision"] < floor


def guarded_recalibrate(router: OpenSetRouter, base_ranked, base_truth, ledger_ranked, ledger_truth, grid=(.05, .1, .15, .2, .3)):
    """Refit the LINK route on base + ledger rows; keep it only if base precision drops < 2 pt and ledger
    precision does not drop. The server swaps the route in place (no champion rebuild)."""
    def prec(rt, ranked, truth):
        ok = [rk[0][0] == t for rk, t in zip(ranked, truth) if rt.route(rk)[0] == "LINK"]
        return sum(ok) / len(ok) if ok else 0.0
    b0, l0 = prec(router, base_ranked, base_truth), prec(router, ledger_ranked, ledger_truth)
    best = router
    for a in grid:
        cand = OpenSetRouter(alpha=a).calibrate(base_ranked + ledger_ranked, base_truth + ledger_truth)
        b1, l1 = prec(cand, base_ranked, base_truth), prec(cand, ledger_ranked, ledger_truth)
        if b1 >= b0 - .02 and l1 > l0:
            best, l0 = cand, l1
    return best


def evaluate(champ, router, cold, runs, unseen=frozenset()):
    m = defaultdict(list)
    for r in runs:
        d = decide(champ, router, cold, r)
        if r["label"] == "NONE":
            m["none_not_linked"].append(d["route"] != "LINK")
            continue
        m["top1"].append(d["top3"][0] == r["label"]); m["top3"].append(r["label"] in d["top3"])
        if d["route"] == "LINK":
            m["link_precision"].append(d["bug"] == r["label"])
        m["link_rate"].append(d["route"] == "LINK")
        if r["label"] in unseen:
            m["unseen_top1"].append(d["top3"][0] == r["label"])
    return {k: (round(100 * sum(v) / len(v), 1), len(v)) for k, v in m.items() if v}, Counter(
        decide(champ, router, cold, r)["route"] for r in runs)

## 1. Train on the past, calibrate on the next window, test on bugs held out by group

In [ ]:
from synthetic import make_world, split
from classifier import *
bugs, runs = make_world(n_bugs=60, n_runs=2000, seed=2)
runs = split(runs, bugs, seed=2)
tr = sorted([r for r in runs if r["split"] == "train"], key=lambda r: r["start"]); test = [r for r in runs if r["split"] == "test"]
train, calib = tr[: int(.8 * len(tr))], tr[int(.8 * len(tr)):]
champ = Champion(K=5).fit(train, bugs)
trained = {r["label"] for r in train}
cb = [r for r in calib if r["label"] != "NONE"]
router = OpenSetRouter(alpha=.1).calibrate([champ.rank(r) for r in cb], [r["label"] for r in cb])
cold = TextColdStart().fit(bugs, trained).calibrate([r for r in calib if r["label"] in trained])
unseen = {r["label"] for r in test} - trained - {"NONE"}
metrics, routes = evaluate(champ, router, cold, test, unseen)
print(metrics); print(routes, "| cold-start theta", round(cold.theta, 3))

## 2. What a person sees: top 3 (never hidden), the route and why

In [ ]:
r = next(r for r in test if r["label"] in trained - {"NONE"})
d = decide(champ, router, cold, r)
print("truth", r["label"]); print({k: d[k] for k in ("route", "top3", "set")}); print("why:", d["why"])

## 3. Override rule on top of an agent verdict (applied only when the agent ran)

In [ ]:
ranked = champ.rank(r)
print(override_rule(ranked, {"bug": ranked[1][0], "confidence": .75}, probe={ranked[1][0]: .5, ranked[0][0]: .2}))
print(override_rule(ranked, {"bug": ranked[1][0], "confidence": .75}, probe={ranked[1][0]: 0}))

## 4. Lifecycle: forward ledger → drift alarm → guarded recalibration

In [ ]:
import random
led = Ledger(); rng = random.Random(0)
fwd = [r for r in test if r["label"] in trained - {"NONE"}][:60]
for x in fwd:
    d = decide(champ, router, cold, x); led.record(x["run"], d["route"], d["bug"])
led.resolve({x["run"]: (x["label"] if rng.random() > .35 else "BUG-999") for x in fwd})   # simulate label drift
print(led.forward(), "recalibrate?", led.needs_recalibration())
new_router = guarded_recalibrate(router, [champ.rank(x) for x in cb], [x["label"] for x in cb],
                                 [champ.rank(x) for x in fwd], [led.rows[x["run"]]["truth"] for x in fwd])
print("alpha", router.alpha, "->", new_router.alpha)
reg = Registry(); print(reg.consider("champion-1", .81, {"C2": .1}), reg.consider("challenger", .80, {"C2": 1}))